In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

catalog = "logistics"

bronze_schema = f"bronze"
silver_schema = f"silver"
gold_schema = f"gold"

print("Catalog:", catalog)
print("Bronze:", bronze_schema)
print("Silver:", silver_schema)
print("Gold:", gold_schema)

In [0]:
tables = {
    "bronze_orders": f"{bronze_schema}.shipment_orders",
    "bronze_events": f"{bronze_schema}.shipment_tracking_events",
    "bronze_attempts": f"{bronze_schema}.delivery_attempts",

    "silver_orders": f"{silver_schema}.shipment_orders",
    "silver_events": f"{silver_schema}.shipment_tracking_events",
    "silver_attempts": f"{silver_schema}.delivery_attempts",
    "silver_carrier": f"{silver_schema}.carrier_master",
    "silver_warehouse": f"{silver_schema}.warehouse_master",
    "silver_route": f"{silver_schema}.route_master",

    "status_history": f"{silver_schema}.silver_shipment_status_history",
    "current_status": f"{silver_schema}.silver_shipment_current",
    "lifecycle": f"{silver_schema}.silver_shipment_lifecycle",
    "lifecycle_exceptions": f"{silver_schema}.silver_lifecycle_exceptions",
    "lifecycle_summary": f"{silver_schema}.silver_shipment_lifecycle_summary",

    "gold_by_status": f"{gold_schema}.gold_shipments_by_status",
    "gold_carrier_perf": f"{gold_schema}.gold_carrier_performance",
    "gold_daily": f"{gold_schema}.gold_daily_shipments",
    "gold_dest_city": f"{gold_schema}.gold_destination_city_summary",
    "gold_service_level": f"{gold_schema}.gold_service_level_summary",
    "gold_attempt_summary": f"{gold_schema}.gold_delivery_attempt_summary",
    "gold_city_success": f"{gold_schema}.gold_city_delivery_success",
    "gold_transitions": f"{gold_schema}.gold_status_transitions",
    "gold_cdc_summary": f"{gold_schema}.gold_cdc_operation_summary",

    "dim_carrier": f"{gold_schema}.dim_carrier",
    "dim_warehouse": f"{gold_schema}.dim_warehouse",
    "dim_route": f"{gold_schema}.dim_route"
}

for key, value in tables.items():
    print(key, ":", value)

In [0]:
#checking if tables exists
existing_tables = []
missing_tables = []

for name, table_name in tables.items():

    # tables values are schema.table; prefix the catalog for a fully qualified name
    table_name = f"{catalog}.{table_name}"

    if spark.catalog.tableExists(table_name):
        existing_tables.append(table_name)
        print("EXISTS:", table_name)

    else:
        missing_tables.append(table_name)
        print("MISSING:", table_name)

print("\nExisting tables:", len(existing_tables))
print("Missing tables:", len(missing_tables))

In [0]:
#Bronze vs Silver record count reconciliation
pairs = [
    ("Shipment Orders",
     tables["bronze_orders"],
     tables["silver_orders"]),

    ("Tracking Events",
     tables["bronze_events"],
     tables["silver_events"]),

    ("Delivery Attempts",
     tables["bronze_attempts"],
     tables["silver_attempts"])
]

reconciliation = []

for dataset, bronze_table, silver_table in pairs:

    bronze_table = f"{catalog}.{bronze_table}"
    silver_table = f"{catalog}.{silver_table}"

    if (spark.catalog.tableExists(bronze_table)
        and spark.catalog.tableExists(silver_table)):

        bronze_count = spark.table(bronze_table).count()
        silver_count = spark.table(silver_table).count()

        reconciliation.append((
            dataset,
            bronze_count,
            silver_count,
            bronze_count - silver_count,
            "MATCH" if bronze_count == silver_count else "REVIEW"
        ))

reconciliation_df = spark.createDataFrame(
    reconciliation,
    [
        "dataset",
        "bronze_count",
        "silver_count",
        "count_difference",
        "status"
    ]
)

display(reconciliation_df)

In [0]:
#Check duplicate records
#check duplicate shipment IDs in Silver Orders.
orders_df = spark.table(f"{catalog}.{tables['silver_orders']}")

duplicate_orders = (
    orders_df
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate shipment IDs:")
display(duplicate_orders)

#check duplicate tracking event IDs.
events_df = spark.table(f"{catalog}.{tables['silver_events']}")

duplicate_events = (
    events_df
    .groupBy("tracking_event_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate tracking event IDs:")
display(duplicate_events)

#duplicate grain keys in the Gold aggregate tables.
gold_grain_keys = {
    "gold_by_status": ["shipment_status"],
    "gold_carrier_perf": ["carrier_id"],
    "gold_daily": ["shipment_date"],
    "gold_dest_city": ["destination_city"],
    "gold_service_level": ["service_level"],
    "gold_attempt_summary": ["attempt_result", "failure_reason"],
    "gold_city_success": ["delivery_city"],
    "gold_transitions": ["from_status", "to_status"],
    "gold_cdc_summary": ["cdc_operation"],
}

gold_duplicate_rows = []
for key, grain in gold_grain_keys.items():
    dup_count = (
        spark.table(f"{catalog}.{tables[key]}")
        .groupBy(*grain)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    gold_duplicate_rows.append((tables[key], ", ".join(grain), dup_count))

duplicate_fact = spark.createDataFrame(
    gold_duplicate_rows,
    "gold_table string, grain string, duplicate_keys long"
)

print("Duplicate grain keys in Gold tables:")
display(duplicate_fact)

In [0]:
#Check missing mandatory values
def null_check(table_name, columns):

    df = spark.table(f"{catalog}.{table_name}")

    expressions = [
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in columns
    ]

    return df.select(*expressions)


print("Shipment Orders:")
display(
    null_check(
        tables["silver_orders"],
        ["shipment_id", "order_id", "customer_id"]
    )
)

print("Tracking Events:")
display(
    null_check(
        tables["silver_events"],
        ["tracking_event_id", "shipment_id", "event_ts", "event_type"]
    )
)


In [0]:
#Validate shipment current state
current_df = spark.table(f"{catalog}.{tables['current_status']}")

duplicate_current = (
    current_df
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate current-state records:")
display(duplicate_current)

print("Total current-state records:",
      current_df.count())

print("Unique shipments:",
      current_df.select("shipment_id").distinct().count())

In [0]:
#Referential integrity checks
#Check missing carrier references
orders_df = spark.table(f"{catalog}.{tables['silver_orders']}")
carrier_df = spark.table(f"{catalog}.{tables['silver_carrier']}")

missing_carriers = (
    orders_df
    .filter(F.col("carrier_id").isNotNull())
    .join(
        carrier_df.select("carrier_id").distinct(),
        on="carrier_id",
        how="left_anti"
    )
)

print("Orders with missing carrier references:")
display(missing_carriers)

#Check missing warehouse references
warehouse_df = spark.table(f"{catalog}.{tables['silver_warehouse']}")

missing_warehouses = (
    orders_df
    .filter(F.col("origin_warehouse_id").isNotNull())
    .join(
        warehouse_df.select("warehouse_id").distinct(),
        orders_df.origin_warehouse_id == warehouse_df.warehouse_id,
        "left_anti"
    )
)

display(missing_warehouses)

In [0]:
#Check Gold aggregate reconciliation against Silver sources
current_cnt = spark.table(f"{catalog}.{tables['current_status']}").count()
history_df = spark.table(f"{catalog}.{tables['status_history']}")
history_cnt = history_df.count()
history_event_cnt = history_df.filter(F.col("cdc_operation") != "NO_EVENT").count()
attempts_cnt = spark.table(f"{catalog}.{tables['silver_attempts']}").count()

# carrier performance is an inner join to dim_carrier, so compare to matched shipments only
carrier_matched_cnt = (
    spark.table(f"{catalog}.{tables['current_status']}")
    .join(spark.table(f"{catalog}.{tables['dim_carrier']}").select("carrier_id"), "carrier_id", "inner")
    .count()
)

def gold_sum(key, col):
    return spark.table(f"{catalog}.{tables[key]}").agg(F.sum(col)).first()[0] or 0

gold_recon_specs = [
    ("gold_by_status", "shipment_count", "silver_shipment_current rows", current_cnt),
    ("gold_daily", "shipment_count", "silver_shipment_current rows", current_cnt),
    ("gold_dest_city", "shipment_count", "silver_shipment_current rows", current_cnt),
    ("gold_service_level", "shipment_count", "silver_shipment_current rows", current_cnt),
    ("gold_carrier_perf", "shipment_count", "current rows matched to dim_carrier", carrier_matched_cnt),
    ("gold_cdc_summary", "shipment_count", "status_history rows", history_cnt),
    ("gold_transitions", "transition_count", "status_history rows with events", history_event_cnt),
    ("gold_attempt_summary", "attempt_count", "silver delivery_attempts rows", attempts_cnt),
    ("gold_city_success", "attempt_count", "silver delivery_attempts rows", attempts_cnt),
]

gold_recon_rows = []
for key, col, source_desc, silver_val in gold_recon_specs:
    gold_val = gold_sum(key, col)
    gold_recon_rows.append((
        tables[key], f"sum({col})", int(gold_val), source_desc, int(silver_val),
        int(silver_val - gold_val),
        "MATCH" if gold_val == silver_val else "REVIEW"
    ))

gold_reconciliation_df = spark.createDataFrame(
    gold_recon_rows,
    "gold_table string, gold_measure string, gold_value long, silver_source string, silver_value long, difference long, status string"
)

display(gold_reconciliation_df)

In [0]:
#Validate Gold metric ranges
#shipment status distribution
display(
    spark.table(f"{catalog}.{tables['gold_by_status']}")
    .orderBy(F.desc("shipment_count"))
)

#success rates must be between 0 and 100
invalid_success_rate = spark.table(f"{catalog}.{tables['gold_city_success']}").filter(
    (F.col("success_rate_pct") < 0) | (F.col("success_rate_pct") > 100)
    | (F.col("successful_attempts") > F.col("attempt_count"))
)
print("City success-rate rows out of range:")
display(invalid_success_rate)

#delivered count cannot exceed shipment count; cost metrics cannot be negative
invalid_carrier_metrics = spark.table(f"{catalog}.{tables['gold_carrier_perf']}").filter(
    (F.col("delivered_count") > F.col("shipment_count"))
    | (F.col("total_revenue") < 0)
    | (F.col("avg_cost_per_kg") < 0)
)
print("Carrier performance rows out of range:")
display(invalid_carrier_metrics)

#CDC operation percentages should add up to ~100
cdc_pct_total = spark.table(f"{catalog}.{tables['gold_cdc_summary']}").agg(F.sum("pct_of_total")).first()[0]
print("CDC pct_of_total sum:", cdc_pct_total)

In [0]:
#Check lifecycle summary
lifecycle_df = spark.table(f"{catalog}.{tables['lifecycle_summary']}")

duplicate_lifecycle = (
    lifecycle_df
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_lifecycle)
#Check negative stage durations (duration lives in the lifecycle table)
lifecycle_detail_df = spark.table(f"{catalog}.{tables['lifecycle']}")

negative_duration = lifecycle_detail_df.filter(
    F.col("stage_duration_hours") < 0
)

display(negative_duration)

In [0]:
#Generate final validation summary
checks = []

def add_check(name, actual, expected, status):

    checks.append((
        name,
        str(actual),
        str(expected),
        status
    ))


# Duplicate shipment IDs
add_check(
    "Duplicate Shipment IDs",
    duplicate_orders.count(),
    0,
    "PASS" if duplicate_orders.count() == 0 else "FAIL"
)

# Duplicate tracking events
add_check(
    "Duplicate Tracking Event IDs",
    duplicate_events.count(),
    0,
    "PASS" if duplicate_events.count() == 0 else "FAIL"
)

# Current state uniqueness
current_duplicates = duplicate_current.count()

add_check(
    "Current State Uniqueness",
    current_duplicates,
    0,
    "PASS" if current_duplicates == 0 else "FAIL"
)

# Gold aggregate grain uniqueness
fact_duplicates = duplicate_fact.agg(F.sum("duplicate_keys")).first()[0] or 0

add_check(
    "Gold Aggregate Grain Uniqueness",
    fact_duplicates,
    0,
    "PASS" if fact_duplicates == 0 else "FAIL"
)

# Gold vs Silver reconciliation
gold_recon_mismatches = gold_reconciliation_df.filter(F.col("status") != "MATCH").count()

add_check(
    "Gold vs Silver Reconciliation Mismatches",
    gold_recon_mismatches,
    0,
    "PASS" if gold_recon_mismatches == 0 else "FAIL"
)

# Gold metric ranges
gold_range_issues = invalid_success_rate.count() + invalid_carrier_metrics.count()

add_check(
    "Gold Metric Range Issues",
    gold_range_issues,
    0,
    "PASS" if gold_range_issues == 0 else "FAIL"
)

# Lifecycle uniqueness
lifecycle_duplicates = duplicate_lifecycle.count()

add_check(
    "Lifecycle Summary Uniqueness",
    lifecycle_duplicates,
    0,
    "PASS" if lifecycle_duplicates == 0 else "FAIL"
)

# Negative durations
negative_count = negative_duration.count()

add_check(
    "Negative Lifecycle Durations",
    negative_count,
    0,
    "PASS" if negative_count == 0 else "FAIL"
)

# Missing carrier references
missing_carrier_count = missing_carriers.count()

add_check(
    "Missing Carrier References",
    missing_carrier_count,
    0,
    "PASS" if missing_carrier_count == 0 else "FAIL"
)

validation_summary = spark.createDataFrame(
    checks,
    [
        "validation_name",
        "actual_result",
        "expected_result",
        "status"
    ]
)

display(validation_summary)

In [0]:
#Save validation results as a Delta table
validation_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalog}.{gold_schema}.shipment_reconciliation_report"
    )

print("Reconciliation report saved successfully!")

In [0]:
%sql
SELECT *
FROM logistics.gold.shipment_reconciliation_report;

In [0]:
# Verify Bronze vs Silver tables — inspect schema changes and record counts

pairs = [
    ("Shipment Orders", f"{catalog}.{tables['bronze_orders']}", f"{catalog}.{tables['silver_orders']}"),
    ("Tracking Events", f"{catalog}.{tables['bronze_events']}", f"{catalog}.{tables['silver_events']}"),
    ("Delivery Attempts", f"{catalog}.{tables['bronze_attempts']}", f"{catalog}.{tables['silver_attempts']}"),
]

for dataset, bronze_tbl, silver_tbl in pairs:
    print("=" * 80)
    print(f"  Dataset: {dataset}")
    print(f"  Bronze : {bronze_tbl}")
    print(f"  Silver : {silver_tbl}")
    print("=" * 80)

    if not (spark.catalog.tableExists(bronze_tbl) and spark.catalog.tableExists(silver_tbl)):
        print("  ⚠️  One or both tables do not exist — skipping\n")
        continue

    bronze_df = spark.table(bronze_tbl)
    silver_df = spark.table(silver_tbl)

    # --- Record counts ---
    bronze_count = bronze_df.count()
    silver_count = silver_df.count()
    diff = bronze_count - silver_count
    status = "✅ MATCH" if diff == 0 else "⚠️  MISMATCH"

    print(f"\n  📊 Record Counts")
    print(f"     Bronze count : {bronze_count:,}")
    print(f"     Silver count : {silver_count:,}")
    print(f"     Difference   : {diff:,}  ({status})")

    # --- Schema changes ---
    bronze_cols = {f.name: str(f.dataType) for f in bronze_df.schema.fields}
    silver_cols = {f.name: str(f.dataType) for f in silver_df.schema.fields}

    added_cols = set(silver_cols.keys()) - set(bronze_cols.keys())
    removed_cols = set(bronze_cols.keys()) - set(silver_cols.keys())
    common_cols = set(bronze_cols.keys()) & set(silver_cols.keys())
    type_changes = {
        c: (bronze_cols[c], silver_cols[c])
        for c in common_cols
        if bronze_cols[c] != silver_cols[c]
    }

    print(f"\n  🔧 Schema Changes")
    print(f"     Bronze columns : {len(bronze_cols)}")
    print(f"     Silver columns : {len(silver_cols)}")

    if added_cols:
        print(f"     ➕ Columns added in Silver   : {sorted(added_cols)}")
    else:
        print(f"     ➕ Columns added in Silver   : None")

    if removed_cols:
        print(f"     ➖ Columns removed in Silver : {sorted(removed_cols)}")
    else:
        print(f"     ➖ Columns removed in Silver : None")

    if type_changes:
        print(f"     🔄 Data type changes:")
        for col, (b_type, s_type) in sorted(type_changes.items()):
            print(f"        • {col}: {b_type} → {s_type}")
    else:
        print(f"     🔄 Data type changes         : None")

    print()

# --- Summary table ---
print("\n" + "=" * 80)
print("  📋 SUMMARY")
print("=" * 80)

summary_rows = []
for dataset, bronze_tbl, silver_tbl in pairs:
    if spark.catalog.tableExists(bronze_tbl) and spark.catalog.tableExists(silver_tbl):
        bc = spark.table(bronze_tbl).count()
        sc = spark.table(silver_tbl).count()
        summary_rows.append((dataset, bc, sc, bc - sc, "MATCH" if bc == sc else "MISMATCH"))

summary_df = spark.createDataFrame(
    summary_rows,
    ["dataset", "bronze_count", "silver_count", "count_difference", "status"]
)
display(summary_df)